# Система детекции аномалий и динамического ценообразования

## Цель

Решение состоит из двух связанных частей:

1. Найти технически подозрительные значения `sales` и показать, какие наблюдаемые факторы объясняют ожидаемый спрос.
2. Построить прогноз спроса на один день вперёд и подобрать цены для трёх разных целей: продажи, GMV и прибыль.

Ключевой принцип решения — **временная корректность**. Все лаги и rolling-статистики используют только прошлые наблюдения. Скрытая колонка `is_anomaly_label` не используется при построении score, признаков или моделей; она нужна только для финальной проверки детектора.

### Основные допущения

- На следующий день известны план промо, рекламный бюджет, текущая цена конкурента и доступный остаток. В демонстрационном сценарии промо выключено, а неизвестные внешние признаки заменены последними устойчивыми значениями.
- `latent_trend_val` исключён: скрытый будущий тренд нельзя гарантированно получить в production.
- `feat_00...feat_12` исключены, поскольку у них нет бизнес-семантики и способа воспроизведения online.
- `stock_level` используется как признак, но не как жёсткий cap: в данных продажи часто превышают остаток, поэтому семантика среза остатка неоднозначна.
- Наблюдательные данные не доказывают причинный эффект цены. Эластичность регуляризуется, ограничивается по знаку, а запуск рекомендаций должен проходить через A/B-тест.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "retail_mle_task_data.parquet").exists():
    PROJECT_DIR = ROOT
elif (ROOT / "ml-interview-task" / "retail_mle_task_data.parquet").exists():
    PROJECT_DIR = ROOT / "ml-interview-task"
else:
    raise FileNotFoundError("Не найден retail_mle_task_data.parquet")

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_DIR / ".mplconfig"))
(PROJECT_DIR / ".mplconfig").mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from retail_pricing.anomaly import detector_metrics, score_anomalies
from retail_pricing.data import data_quality_report, load_dataset
from retail_pricing.demand import fit_final_demand_model, walk_forward_validation
from retail_pricing.elasticity import estimate_elasticities
from retail_pricing.explainability import root_cause_report
from retail_pricing.features import build_model_frame
from retail_pricing.optimization import optimize_prices

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = PROJECT_DIR / "retail_mle_task_data.parquet"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
SEED = 42

## 1. Аудит данных

Перед моделированием проверим полноту панели, пропуски, дубликаты и бизнес-ограничения. Проверка `sales_lag_1` сравнивает готовую колонку с фактическим сдвигом ряда; при расхождениях далее всё равно используются собственные причинные лаги.

In [ ]:
dataset = load_dataset(DATA_PATH)
quality = data_quality_report(dataset)

print(f"Период: {dataset['date'].min().date()} — {dataset['date'].max().date()}")
print(f"Форма: {dataset.shape}; товаров: {dataset['item_id'].nunique()}")
display(quality)

category_summary = (
    dataset.groupby("category")
    .agg(
        items=("item_id", "nunique"),
        rows=("item_id", "size"),
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
        mean_price=("price", "mean"),
        promo_rate=("promo_active", "mean"),
    )
    .sort_values("mean_sales", ascending=False)
)
display(category_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(np.log1p(dataset["sales"]), bins=70, color="#287271", alpha=0.85)
axes[0].set_title("Распределение log(1 + sales)")
axes[0].set_xlabel("log(1 + sales)")

dataset.groupby("category")["sales"].median().sort_values().plot(
    kind="barh", ax=axes[1], color="#D98C4A"
)
axes[1].set_title("Медианные продажи по категориям")
axes[1].set_xlabel("sales")
plt.tight_layout()
plt.show()

## 2. Детекция аномалий

Для каждого товара вычисляется локальное ожидание по предыдущим 28 дням. Основной score:

$$
score_{i,t}=\frac{|y_{i,t}-\mu_{i,t}^{(28)}|}
{\sigma_{i,t}^{(28)}+\sqrt{\mu_{i,t}^{(28)}+1}}.
$$

Добавка $\sqrt{\mu+1}$ стабилизирует score на малых продажах. Положительный порог задаётся 99.9-м перцентилем `spike_score`, рассчитанным только по датам, предшествующим текущей. Обычный всплеск в промо-день считается объяснимым бизнес-событием; в high-confidence алерт он попадает только при превышении порога в 10 раз. Для провалов есть отдельное правило review-уровня: ноль считается подозрительным, если локальное среднее не меньше 15 и товар продавался более чем в 90% предыдущих дней.

Получаются два уровня: высокоточные технические всплески для немедленного алерта и более широкий список review-кандидатов. Для общего ranking промо-всплески понижаются, а неожиданные нули поднимаются до уровня текущего порога.

In [ ]:
scored = score_anomalies(dataset)
detector_report = detector_metrics(scored)

print(f"Порог spike_score на последнюю дату: {scored.attrs['latest_spike_threshold']:.3f}")
print(f"High-confidence алертов: {scored['is_high_confidence_anomaly'].sum():,}")
print(f"Всего review-кандидатов: {scored['is_candidate_anomaly'].sum():,}")
display(detector_report)

detected = scored[scored["is_candidate_anomaly"]].sort_values(
    "ranking_score", ascending=False
)
display(
    detected[
        [
            "date",
            "item_id",
            "sales",
            "local_mean_28",
            "ranking_score",
            "anomaly_type",
        ]
    ].head(20)
)

In [ ]:
spike_event = detected[detected["anomaly_type"].eq("technical_spike")].iloc[0]
drop_event = (
    detected[detected["anomaly_type"].eq("unexpected_zero_review")]
    .sort_values("local_mean_28", ascending=False)
    .iloc[0]
)
events = [spike_event, drop_event]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for ax, event in zip(axes, events):
    item = event["item_id"]
    center = event["date"]
    part = scored[
        scored["item_id"].eq(item)
        & scored["date"].between(center - pd.Timedelta(days=45), center + pd.Timedelta(days=20))
    ]
    ax.plot(part["date"], part["sales"], label="sales", color="#2A5C82")
    ax.plot(part["date"], part["local_mean_28"], label="past-only mean", color="#D98C4A")
    ax.scatter([center], [event["sales"]], color="#B33A3A", s=70, zorder=3, label="alert")
    ax.set_title(f"{item}: {event['anomaly_type']}")
    ax.legend()

plt.tight_layout()
plt.show()

### Интерпретация качества

`Average Precision` и `precision@K` уместнее accuracy: положительный класс крайне редок. Метрики считаются по контекстному `ranking_score`. Отдельно показываются precision/recall двух уровней: `high_confidence` и `review`. Скрытая разметка содержит также нулевые точки в прерывистых низкочастотных рядах; без журнала остатков и ETL-сбоев их нельзя надёжно отделить от нормальных нулей.

После детекции подозрительные значения заменяются локальной медианой **только при построении последующих лагов**. Это предотвращает каскад ложных прогнозов после технического всплеска. Сами строки-кандидаты исключаются из обучения спроса.

## 3. Модель спроса и временная валидация

Используется одна глобальная XGBoost-модель на все товары. Общая модель делит статистическую силу между рядами, а `item_id` и `category` сохраняют товарную неоднородность. Цель моделируется как `log1p(sales)`, что снижает влияние тяжёлого хвоста.

Признаки:

- лаги, rolling median/mean, EWM и среднее по тому же дню недели;
- цена, себестоимость и относительная цена конкурента;
- исторический средний спрос в текущем promo-режиме и глубина скидки относительно 90-дневной медианы;
- промо и рекламный бюджет;
- остаток, погода и календарь;
- идентификатор товара и категория.

Валидация — три expanding-window фолда по 90 дней. В таблице показаны метрики как на всех строках, так и на строках, не помеченных нашим детектором.

In [ ]:
model_frame = build_model_frame(scored)
validation = walk_forward_validation(model_frame)

validation_summary = (
    validation.metrics.groupby(["model", "scope"])[["MAE", "RMSE", "WAPE", "SMAPE"]]
    .mean()
    .sort_values("WAPE")
)
display(validation.metrics)
display(validation_summary)

In [ ]:
last_fold = validation.predictions["fold_start"].max()
plot_data = validation.predictions[validation.predictions["fold_start"].eq(last_fold)]
plot_items = (
    plot_data.groupby("item_id", observed=True)["sales"].mean().sort_values().iloc[[50, 150]].index
)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, item_id in zip(axes, plot_items):
    part = plot_data[plot_data["item_id"].eq(item_id)].sort_values("date")
    ax.plot(part["date"], part["sales"], label="actual", linewidth=1.6)
    ax.plot(part["date"], part["prediction"], label="forecast", linewidth=1.6)
    ax.set_title(item_id)
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Root Cause Analysis

Модель обучается заново на всей доступной истории без строк-кандидатов. Используется встроенный TreeSHAP XGBoost (`pred_contribs=True`), поэтому отдельная библиотека `shap` не требуется. Вклады агрегируются по бизнес-группам: история, цена, промо, конкуренция, доступность и календарные/внешние факторы.

Важно различать два вопроса:

1. **Что сформировало ожидаемый спрос?** На него отвечает крупнейший контекстный вклад.
2. **Объясняет ли контекст фактическую аномалию?** Если остаток на log-шкале очень велик, причиной считается `unexplained_residual`, а не фактор с самым большим SHAP.

Иными словами, SHAP объясняет прогноз модели, но не превращает технический выброс в бизнес-событие.

In [ ]:
final_model = fit_final_demand_model(model_frame)
top_alerts = model_frame[model_frame["is_candidate_anomaly"]].nlargest(30, "ranking_score")
root_causes = root_cause_report(final_model, model_frame, top_alerts)

display(root_causes.head(20))
root_causes.to_csv(ARTIFACT_DIR / "root_cause_report.csv", index=False)

## 5. Эластичность и динамическое ценообразование

Для товара $i$ используется локальная постоянная эластичность:

$$
\hat q_i(p)=\hat q_i(p_0)\left(\frac{p}{p_0}\right)^{\varepsilon_i}.
$$

`raw_elasticity` оценивается Ridge-регрессией на log-шкале с контролями промо, конкурента, рекламы, лагов, погоды и календаря. Из-за сильного observational confounding цена–промо часть сырых коэффициентов имеет неверный положительный знак. Поэтому автоматизированная политика использует:

- ограничение $-5 \le \varepsilon_i \le -0.05$;
- shrinkage к медиане категории, причём слабая вариативность цены усиливает shrinkage;
- сохранение сырой оценки для диагностики и дальнейшего эксперимента;
- маркировку `item_supported`/`category_prior_driven`: если знак или качество индивидуальной оценки ненадёжны, допустимое изменение цены дополнительно ограничивается 5%.

Это консервативная инженерная мера, а не замена причинному A/B-тесту.

In [ ]:
elasticity = estimate_elasticities(model_frame)
elasticity_summary = (
    elasticity.groupby("price_segment", observed=True)
    .agg(
        items=("item_id", "size"),
        raw_median=("raw_elasticity", "median"),
        production_median=("elasticity", "median"),
        median_r2=("model_r2", "median"),
        prior_driven_share=("elasticity_source", lambda values: values.eq("category_prior_driven").mean()),
    )
    .sort_index()
)
display(elasticity_summary)

fig, ax = plt.subplots(figsize=(10, 4))
elasticity.boxplot(column="elasticity", by="price_segment", ax=ax, grid=False)
ax.set_title("Регуляризованная эластичность по ценовым сегментам")
ax.set_xlabel("Ценовой сегмент товара")
ax.set_ylabel("Эластичность")
plt.suptitle("")
plt.tight_layout()
plt.show()

### Цели и ограничения

Оптимизируются три разные функции:

$$Sales(p)=\hat q(p),$$
$$GMV(p)=p\hat q(p),$$
$$Profit(p)=(p-cost)\hat q(p).$$

Продажи почти всегда предпочитают нижнюю допустимую цену, поэтому нельзя выдавать один оптимум за решение всех бизнес-задач. Результат содержит три цены и позволяет бизнесу выбрать политику.

Guardrails:

- цена не ниже `cost × 1.05`;
- изменение не больше 15% от последней регулярной цены, а для `category_prior_driven` оценок — не больше 5%; исключение — текущая цена уже ниже минимальной маржи, тогда она поднимается до `cost × 1.05` со статусом `margin_floor_correction`;
- диапазон 5–95% исторических регулярных цен применяется, когда он совместим с лимитом изменения; если текущая цена уже вне диапазона и пересечение пусто, цена удерживается со статусом `historical_range_conflict_hold`;
- при прогнозе спроса меньше одной единицы цена удерживается около текущей (`low_demand_hold`), поскольку целевая функция практически плоская; минимальная маржа при этом остаётся обязательной;
- оптимизация выполняется сеточным поиском, поэтому поведение легко проверить и объяснить.

In [ ]:
recommendations, price_curves = optimize_prices(
    final_model,
    model_frame,
    elasticity,
    price_change_limit=0.15,
    minimum_margin=0.05,
)

recommendations.to_csv(ARTIFACT_DIR / "price_recommendations.csv", index=False)
elasticity.to_csv(ARTIFACT_DIR / "elasticity_estimates.csv", index=False)

display(recommendations.head(20))
print("Среднее абсолютное изменение profit-optimal цены:",
      np.mean(np.abs(recommendations["profit_optimal_price"] /
                     recommendations["current_regular_price"] - 1)))

In [ ]:
high_volume_item = recommendations.nlargest(1, "base_demand")["item_id"].iloc[0]
elastic_item = recommendations.nsmallest(1, "elasticity")["item_id"].iloc[0]
plot_items = list(dict.fromkeys([high_volume_item, elastic_item]))

fig, axes = plt.subplots(len(plot_items), 1, figsize=(12, 4 * len(plot_items)), squeeze=False)
for ax, item_id in zip(axes.ravel(), plot_items):
    curve = price_curves[price_curves["item_id"].eq(item_id)]
    rec = recommendations[recommendations["item_id"].eq(item_id)].iloc[0]
    ax.plot(curve["candidate_price"], curve["predicted_profit"], color="#287271", linewidth=2)
    ax.axvline(rec["current_regular_price"], color="#D98C4A", linestyle="--", label="current regular")
    ax.axvline(rec["profit_optimal_price"], color="#B33A3A", linestyle="--", label="profit optimum")
    ax.set_title(f"Profit curve: {item_id}")
    ax.set_xlabel("Цена")
    ax.set_ylabel("Прогноз прибыли")
    ax.legend()

plt.tight_layout()
plt.show()

## Выводы и production-план

1. Причинный rolling-score хорошо отделяет крупные технические всплески. Неожиданные нули требуют отдельного правила и журнала доступности товара/ETL-сбоев.
2. Контекст промо отделяет реальные бизнес-всплески от технических ошибок и резко сокращает число ложных high-confidence алертов.
3. Дополнительные причинные лаги, EWM, недельная сезонность и promo-history улучшают глобальную табличную модель относительно rolling baseline.
4. Root-cause отчёт намеренно разделяет контекстные драйверы ожидаемого спроса и необъяснённый остаток.
5. Оптимальные цены для продаж, GMV и прибыли различаются. В API следует явно передавать выбранную бизнес-цель.
6. Offline-оптимизация не доказывает финансовый uplift. Перед автоматическим rollout нужны shadow mode, ограниченный A/B-тест, мониторинг маржи, WAPE, доли guardrail-срабатываний и drift эластичности.

Созданные артефакты:

- `artifacts/root_cause_report.csv`;
- `artifacts/elasticity_estimates.csv`;
- `artifacts/price_recommendations.csv`.